# Prework

## Functions and libraries

In [1]:
import os
# PATH = "/Users/luanagiusto/TP-1-ML"  # Cambia esto si tu path es diferente
PATH = "C:/Users/julia/ML_TP"

In [2]:
import pandas as pd
import numpy as np
# from ydata_profiling import ProfileReport
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)
import gc
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
# Nota: Antes de ejecutar este notebook, instala los requisitos con:
# !pip install -r requirements.txt

In [3]:
def data_profiling(df, output_file):
    # Opciones para que sea liviano
    profile = ProfileReport(
        df.sample(20000, random_state=42) if len(df) > 20000 else df,
        title=output_file,
        minimal=True,         # desactiva análisis costosos
        explorative=True      # agrega secciones útiles
    )

    profile.to_file(output_file)  # <-- abre este HTML en el navegador

In [4]:
# Funcion para mostrar un resumen del dataframe
def df_info_summary(df: pd.DataFrame):
    total = len(df)
    non_null = df.notnull().sum()
    nulls = df.isnull().sum()
    dtypes = df.dtypes
    
    resumen = pd.DataFrame({
        "Non-Null Count": non_null,
        "Null Count": nulls,
        "% Null": (nulls / total * 100).round(2),
        "Dtype": dtypes
    })
    print(resumen)

In [5]:
def resumir_por_id(df, id_col='ID', excluir_cols=None, verbose=False, nombre_conteo='n_registros'):
    """
    Sumariza un DataFrame agrupando por una columna ID.
    Calcula métricas estadísticas básicas para columnas numéricas,
    excluyendo las que se indiquen. Incluye conteo total de registros por ID.

    Parámetros:
    - df: DataFrame de entrada con múltiples registros por ID.
    - id_col: nombre de la columna que identifica cada entidad única.
    - excluir_cols: lista de columnas a excluir del resumen (opcional).
    - verbose: si True, imprime columnas incluidas y excluidas.
    - nombre_conteo: nombre de la columna que indica cantidad de registros por ID.

    Retorna:
    - DataFrame con una fila por ID y métricas estadísticas por columna.
    """
    if excluir_cols is None:
        excluir_cols = []

    excluir_set = set(excluir_cols)
    if id_col in excluir_set:
        excluir_set.remove(id_col)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    cols_a_resumir = [col for col in numeric_cols if col not in excluir_set and col != id_col]

    if verbose:
        print(f"Columnas excluidas: {sorted(excluir_set)}")
        print(f"Columnas resumidas: {sorted(cols_a_resumir)}")

    # Agregaciones estadísticas
    agg_funcs = ['mean', 'min', 'max', 'median', 'sum']
    agg_dict = {col: agg_funcs for col in cols_a_resumir}

    # Agregar conteo de registros por ID
    df[nombre_conteo] = 1
    agg_dict[nombre_conteo] = ['count']

    resumen = df.groupby(id_col).agg(agg_dict)
    resumen.columns = [f"{col}_{stat}" for col, stat in resumen.columns]
    resumen = resumen.reset_index()

    return resumen

In [6]:
# Función para limpiar nombres de columnas
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[^a-z0-9_]+', '_', regex=True)
          .str.replace(r'__+', '_', regex=True)
          .str.strip('_')
    )
    return df

In [7]:
# Función auxiliar
def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Evaluación de modelos
def fit_transform_model(train_df):
    TARGET_COL = "target"    # ajustá al nombre de tu columna objetivo
    SAMPLE_FRAC = 0.05       # 5% de las filas

    X = train_df.drop(columns=TARGET_COL)
    y = train_df[TARGET_COL]

    # Muestreo
    sampled_X = X.sample(frac=SAMPLE_FRAC, random_state=42)
    sampled_y = y.loc[sampled_X.index]

    # División en entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = [
        LinearRegression(n_jobs=-1)
    ]   

    for model in models:
        model_name = model.__class__.__name__
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, predictions)
        print(f"{model_name} RMSE: {rmse:.4f}")

## Data import and overview

In [8]:
df = pd.read_parquet(os.path.join(PATH, "adq_datos_output.parquet"), engine='fastparquet')

In [9]:
df_info_summary(df)

                                                    Non-Null Count  Null Count  % Null    Dtype
SK_ID_CURR                                                  356255           0    0.00    int64
TARGET                                                      307511       48744   13.68  float64
NAME_CONTRACT_TYPE                                          356255           0    0.00   object
CODE_GENDER                                                 356255           0    0.00   object
FLAG_OWN_CAR                                                356255           0    0.00   object
FLAG_OWN_REALTY                                             356255           0    0.00   object
CNT_CHILDREN                                                356255           0    0.00    int64
AMT_INCOME_TOTAL                                            356255           0    0.00  float64
AMT_CREDIT                                                  356255           0    0.00  float64
AMT_ANNUITY                             

In [10]:
# Por ahora reemplazo nan con ceros, pero habría que ver si se puede mejorar
cols_to_fill = [c for c in df.columns if c != 'TARGET']
df[cols_to_fill] = df[cols_to_fill].fillna(0)
print("Columnas con valores NaN despues de rellenar:")
print(df.columns[df.isna().any()].tolist())

Columnas con valores NaN despues de rellenar:
['TARGET']


In [11]:
df_info_summary(df)

                                                    Non-Null Count  Null Count  % Null    Dtype
SK_ID_CURR                                                  356255           0    0.00    int64
TARGET                                                      307511       48744   13.68  float64
NAME_CONTRACT_TYPE                                          356255           0    0.00   object
CODE_GENDER                                                 356255           0    0.00   object
FLAG_OWN_CAR                                                356255           0    0.00   object
FLAG_OWN_REALTY                                             356255           0    0.00   object
CNT_CHILDREN                                                356255           0    0.00    int64
AMT_INCOME_TOTAL                                            356255           0    0.00  float64
AMT_CREDIT                                                  356255           0    0.00  float64
AMT_ANNUITY                             

In [12]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 663 entries, SK_ID_CURR to CCB_credit_card_balance_records_count
dtypes: bool(1), float64(606), int64(40), object(16)
memory usage: 1.8+ GB


In [13]:
# OHE de columnas categóricas
# Identificar columnas categóricas
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("Columnas categóricas:", cat_cols)

# Aplicar One Hot Encoding
df = pd.get_dummies(df, columns=cat_cols, dummy_na=True)

df.shape

Columnas categóricas: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


(356255, 809)

In [14]:
# Limpiar nombres de columnas
df = clean_column_names(df)
df_info_summary(df)   

                                                    Non-Null Count  Null Count  % Null    Dtype
sk_id_curr                                                  356255           0    0.00    int64
target                                                      307511       48744   13.68  float64
cnt_children                                                356255           0    0.00    int64
amt_income_total                                            356255           0    0.00  float64
amt_credit                                                  356255           0    0.00  float64
amt_annuity                                                 356255           0    0.00  float64
amt_goods_price                                             356255           0    0.00  float64
region_population_relative                                  356255           0    0.00  float64
days_birth                                                  356255           0    0.00    int64
days_employed                           

In [15]:
# Testeo fit
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2618


## Feature engineering

In [16]:
# Ratios credito-ingreso

# credit_term ≈ meses del crédito
df["credit_term"] = (
    df["amt_credit"]
      .div(df["amt_annuity"].replace(0, np.nan))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# credit_to_income = credito / ingreso total
df["credit_to_income"] = (
    df["amt_credit"]
      .div(df["amt_income_total"].replace(0, np.nan))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# annuity_income_pct = cuota / ingreso total
df["annuity_income_pct"] = (
    df["amt_annuity"]
      .div(df["amt_income_total"].replace(0, np.nan))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# goods_price_to_credit = precio bienes / credito
df["goods_price_to_credit"] = (
    df["amt_goods_price"]
      .div(df["amt_credit"].replace(0, np.nan))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# income_per_child = income / cant hijos
df["income_per_child"] = (
    df["amt_income_total"]
      .div(df["cnt_children"].replace(0, 1))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# income_per_family_member = income / cant miembros de familia
df["income_per_family_member"] = (
    df["amt_income_total"]
      .div(df["cnt_fam_members"].replace(0, 1))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

In [17]:
df_train = df[df["is_test"] == False].drop(columns="is_test")


In [18]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2619


In [19]:
# Edad y antiguedad en la empresa

# age_years = edad en años
# Asegurar columnas base
if "age_years" not in df and "days_birth" in df:
    df["age_years"] = (
        (-df["days_birth"] / 365)
          .replace([np.inf, -np.inf], np.nan)
          .clip(lower=0)
          .fillna(0)
    )

if "emp_years" not in df and "days_employed" in df:
    df["emp_years"] = (
        (-df["days_employed"] / 365)
          .replace([np.inf, -np.inf], np.nan)
          .clip(lower=0, upper=60)
          .fillna(0)
    )

# Relación entre años de empleo y edad
df["emp_to_age"] = (
    df["emp_years"]
      .div(df["age_years"].replace(0, np.nan))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

In [20]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2619


In [21]:
# Suma y ratios de flags
df["flag_sum"] = df.filter(like='flag_').sum(axis=1)    
df['mismatch_address_flags'] = df[[
    'reg_region_not_live_region',
    'reg_region_not_work_region',
    'live_region_not_work_region',
    'reg_city_not_live_city',
    'reg_city_not_work_city',
    'live_city_not_work_city'
]].sum(axis=1)

In [22]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2619


In [23]:
# external_sources_mean = media de external_sources
df["external_sources_mean"] = df[[
    'ext_source_1',
    'ext_source_2',
    'ext_source_3'
]].mean(axis=1)

df["external_sources_mean_2_3"] = df[[
    'ext_source_2',
    'ext_source_3'
]].mean(axis=1)

In [24]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2619


In [25]:
# ext_source_prod
df["ext_source_prod"] = (
    df["ext_source_1"] *
    df["ext_source_2"] *
    df["ext_source_3"]
).fillna(0)

df["ext_source_prod_2x3"] = (
    df["ext_source_2"] *
    df["ext_source_3"]
).fillna(0)

df["ext_source_prod_1x3"] = (
    df["ext_source_1"] *
    df["ext_source_3"]
).fillna(0)

df["ext_source_prod_1x2"] = (
    df["ext_source_1"] *
    df["ext_source_2"]
).fillna(0)

In [26]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2615


In [27]:
# Características de Vivienda y Edificio
for suf in ['avg','mode','medi']:
    # living / apartments
    df[f'living_area_ratio_{suf}'] = (
        df[f'livingarea_{suf}']
        .div(df[f'apartments_{suf}'].replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    # nonliving / nonlivingapartments
    df[f'nonliving_area_ratio_{suf}'] = (
        df[f'nonlivingarea_{suf}']
        .div(df[f'nonlivingapartments_{suf}'].replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    # diferencia de pisos
    df[f'floor_diff_{suf}'] = (
        df[f'floorsmax_{suf}'] - df[f'floorsmin_{suf}']
    )
    # commonarea / entrances
    df[f'area_per_entrance_{suf}'] = (
        df[f'commonarea_{suf}']
        .div(df[f'entrances_{suf}'].replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    # elevators / entrances
    df[f'elevators_per_entrance_{suf}'] = (
        df[f'elevators_{suf}']
        .div(df[f'entrances_{suf}'].replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

In [28]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2615


In [29]:
# Variables de Círculo Social
df['def30_rate'] = (
    df['def_30_cnt_social_circle']
    .div(df['obs_30_cnt_social_circle'].replace(0, np.nan))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

df['def60_rate'] = (
    df['def_60_cnt_social_circle']
    .div(df['obs_60_cnt_social_circle'].replace(0, np.nan))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

df['social_obs_total'] = (
    df['obs_30_cnt_social_circle'] + df['obs_60_cnt_social_circle']
)

df['social_def_total'] = (
    df['def_30_cnt_social_circle'] + df['def_60_cnt_social_circle']
)

In [30]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2615


In [31]:
# Consultas al Bureau de Crédito
bureau_cols = [
    'amt_req_credit_bureau_hour',
    'amt_req_credit_bureau_day',
    'amt_req_credit_bureau_week',
    'amt_req_credit_bureau_mon',
    'amt_req_credit_bureau_qrt',
    'amt_req_credit_bureau_year'
]

df['bureau_req_total'] = df[bureau_cols].sum(axis=1)

df['bureau_hour_day_ratio'] = (
    df['amt_req_credit_bureau_hour']
    .div(df['amt_req_credit_bureau_day'].replace(0, np.nan))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

df['bureau_mon_year_ratio'] = (
    df['amt_req_credit_bureau_mon']
    .div(df['amt_req_credit_bureau_year'].replace(0, np.nan))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [32]:
df_train = df[df["is_test"] == False].drop(columns="is_test")
fit_transform_model(df_train)

LinearRegression RMSE: 0.2615


## Guardar df final

In [33]:
# Guardar df final en formato parquet
df_train = df[df["is_test"] == False].drop(columns="is_test")
df_test = df[df["is_test"] == True].drop(columns="is_test")
df_train.to_parquet(os.path.join(PATH, "prework_train_output.parquet"))
df_test.to_parquet(os.path.join(PATH, "prework_test_output.parquet"))


## Oversampling de target

In [34]:
def oversample_only_train(df, target_col="target", test_size=0.2, random_state=42):
    # Split original
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    print("Conteo original TRAIN:", sorted(Counter(y_train).items()))
    print("Conteo original TEST :", sorted(Counter(y_test).items()))

    # Oversampling solo en TRAIN
    ros = RandomOverSampler(random_state=random_state)
    X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

    print("Conteo oversampleado :", sorted(Counter(y_train_res).items()))

    # Reconstruir DataFrame combinado
    df_train_res = pd.DataFrame(X_train_res, columns=X.columns)
    df_train_res[target_col] = y_train_res.values

    df_test = pd.DataFrame(X_test, columns=X.columns)
    df_test[target_col] = y_test.values

    df_combined = pd.concat([df_train_res, df_test], ignore_index=True)
    return df_combined

In [35]:
df_train_os = oversample_only_train(df_train, target_col="target", test_size=0.2, random_state=42)

Conteo original TRAIN: [(0.0, 226148), (1.0, 19860)]
Conteo original TEST : [(0.0, 56538), (1.0, 4965)]
Conteo oversampleado : [(0.0, 226148), (1.0, 226148)]


C:\Users\julia\AppData\Local\Temp\ipykernel_24420\220642653.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_res[target_col] = y_train_res.values


In [36]:
fit_transform_model(df_train_os)

LinearRegression RMSE: 0.4411


In [37]:
# En terminos de RMSE no se observa una mejora, entonces vamos a probar con random forest evaluar en terminos de ROC-AUC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
def fit_transform_rf(X, y):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        n_jobs=-1, 
        random_state=42
    )

    rf.fit(X_train, y_train)

    proba = rf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, proba)
    return rf, auc, proba

In [38]:
# Separar features y target
xs = df_train.drop(columns='target')
y = df_train['target']

# Entrenar y evaluar
rf, auc, proba = fit_transform_rf(xs, y)
print(f"ROC-AUC (sin oversampling): {auc:.6f}")

ROC-AUC (sin oversampling): 0.749565


In [39]:
# Separar features y target
xs = df_train_os.drop(columns='target')
y = df_train_os['target']

# Entrenar y evaluar
rf, auc, proba = fit_transform_rf(xs, y)
print(f"ROC-AUC (con oversampling): {auc:.6f}")

ROC-AUC (con oversampling): 0.831207


## Guardar df oversampleado

In [40]:
# Vemos una mejora con oversampling, entonces guardamos el df oversampleado
df_train_os.to_parquet(os.path.join(PATH, "prework_train_output_oversampled.parquet"))